# HIT Eulerian calculation only

Run these cells in order. This notebook only calculates and validates the fixed-grid Eulerian reference. It does not load particle results or draw comparison figures.


In [1]:
import sys
import importlib
import inspect
from pathlib import Path
import numpy as np

code_dir = Path("/meddy/simingzhang/Analysis/python/python3")
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

import third_order_diagnostics as tod
import hit_eulerian_matching as hem
importlib.invalidate_caches()
tod = importlib.reload(tod)
hem = importlib.reload(hem)

print("module:", hem.__file__)
print("runner:", inspect.signature(hem.run_hit_eulerian))
print("calculator:", inspect.signature(hem.calculate_eulerian_homogeneity))

import sys
import types
import importlib
from pathlib import Path

code_dir = Path(
    "/meddy/simingzhang/Analysis/python/python3"
)

module_path = (
    code_dir
    / "hit_eulerian_matching.py"
)

print("Loading source from:")
print(module_path)

source = module_path.read_text(
    encoding="utf-8"
)

# Confirm the uploaded source really contains MBB
required_source_strings = [
    "do_moving_block_bootstrap",
    "moving_time_block_bootstrap",
    "D3_time_sum",
    "D3_mbb_ci_low",
]

for text in required_source_strings:

    if text not in source:
        raise RuntimeError(
            f"Uploaded module is old: "
            f"'{text}' was not found in "
            f"{module_path}"
        )

# Force execution of the exact source file
hem = types.ModuleType(
    "hit_eulerian_matching"
)

hem.__file__ = str(
    module_path
)

exec(
    compile(
        source,
        str(module_path),
        "exec",
    ),
    hem.__dict__,
)

sys.modules[
    "hit_eulerian_matching"
] = hem

print(
    "MBB runner loaded:",
    hasattr(
        hem,
        "run_hit_eulerian",
    ),
)

print(
    inspect.signature(
        hem.calculate_eulerian_homogeneity
    )
)


module: /meddy/simingzhang/Analysis/python/python3/hit_eulerian_matching.py
runner: (cfg)
calculator: (u, v, r_values, dx, dy=None, nblock_y=4, nblock_x=4, periodic=False, directions=None, min_samples_per_block=500, min_valid_blocks=None, time_chunk=50, require_same_block=False, reservoir_size=100000, random_seed=12345, do_moving_block_bootstrap=False, snapshot_dt=None, mbb_num_boot=1000, mbb_confidence=0.95, mbb_block_length_time=None, mbb_random_seed=271828, show_progress=True, progress_name='Eulerian homogeneity')
Loading source from:
/meddy/simingzhang/Analysis/python/python3/hit_eulerian_matching.py
MBB runner loaded: True
(u, v, r_values, dx, dy=None, nblock_y=4, nblock_x=4, periodic=False, directions=None, min_samples_per_block=500, min_valid_blocks=None, time_chunk=50, require_same_block=False, reservoir_size=100000, random_seed=12345, do_moving_block_bootstrap=False, snapshot_dt=None, mbb_num_boot=1000, mbb_confidence=0.95, mbb_block_length_time=None, mbb_random_seed=271828, s

## Configuration

The separation vector below must also be used when recomputing N=200, 2,000, and 5,000. The upper limit π keeps all eight Eulerian directions inside the minimum-image range.


In [8]:
dx_hit = 2.0 * np.pi / 512.0

# This exact array must later be reused by all particle cases.
r_requested_hit = np.geomspace(
    2.0 * dx_hit,
    np.pi,
    12,
)

cfg_eulerian = dict(
    grid_dir=Path("/meddy/simingzhang/Data/HIT2D/2dt2"),
    timerange_matlab=np.arange(50, 151),
    file_offset=4000,
    r_requested=r_requested_hit,
    dx=dx_hit,
    dy=dx_hit,
    nblock_x=4,
    nblock_y=4,
    min_samples_per_block=1,
    min_valid_blocks=8,
    time_chunk=10,
    require_same_block=False,

    # Physical interval between consecutive HIT snapshots.
    snapshot_dt=0.1,

    # Moving-time-block bootstrap.
    do_moving_block_bootstrap=True,
    mbb_num_boot=1000,
    mbb_confidence=0.95,
    mbb_block_length_time=None,
    mbb_random_seed=271828,
    third_pdf_reservoir_size=100_000,
    random_seed=12345,
    show_progress=True,
    output_path=Path("/meddy/simingzhang/Data/HIT2D/2dt2/HIT_Eulerian_r2dx_to_pi_12scales_t50_150.mat"),
)

print("dx =", dx_hit)
print("requested r =", r_requested_hit)
print("number of times =", len(cfg_eulerian["timerange_matlab"]))
print("Eulerian source files = 4049 ... 4149")


dx = 0.01227184630308513
requested r = [0.02454369 0.03815087 0.05930195 0.09217933 0.14328412 0.22272174
 0.34620008 0.53813557 0.83648132 1.30023181 2.0210885  3.14159265]
number of times = 101
Eulerian source files = 4049 ... 4149


## Calculate

This reads all fixed-grid velocity fields and uses all available grid increments. It does not force Eulerian sample counts to equal particle sample counts.


In [9]:
result_eulerian, output_eulerian = hem.run_hit_eulerian(cfg_eulerian)
print("Saved:", output_eulerian)


Load HIT Eulerian fields: 100%|█████████████████████████████| 101/101 [00:02<00:00, 39.04field/s]


HIT Eulerian reference
  velocity shape: (101, 512, 512)
  requested scales: 12
  directions: 8
  spatial blocks: 4 x 4
  time chunk: 10
  periodic: True
  require same block: False


HIT Eulerian reference: 100%|█████████| 1056/1056 [07:55<00:00,  2.22chunk/s, r=3.14, dir=(-1,1)]



Completed: HIT Eulerian reference
  H_Dl: 12/12 valid scales
  H_Dt: 12/12 valid scales
  H_Dll: 12/12 valid scales
  H_Dtt: 12/12 valid scales
  H_D2: 12/12 valid scales
  H_Dlll: 12/12 valid scales
  H_Dltt: 12/12 valid scales
  H_D3: 12/12 valid scales
Saved Eulerian result:
/meddy/simingzhang/Data/HIT2D/2dt2/HIT_Eulerian_r2dx_to_pi_12scales_t50_150.mat
Saved: /meddy/simingzhang/Data/HIT2D/2dt2/HIT_Eulerian_r2dx_to_pi_12scales_t50_150.mat


## Validate output

Do not proceed to particle comparisons until this cell passes.


In [10]:
required = [
    "r_requested", "r_actual", "dist_axis",
    "count_total", "pair_count",
    "D2", "D3", "H3",
    "third_count", "third_positive_count", "third_negative_count",
    "third_relative_standard_error", "third_cancellation_ratio",
    "dl_reservoir",
    "D3_mbb", "D3_mbb_mean", "D3_mbb_stderr",
    "D3_mbb_ci_low", "D3_mbb_ci_high",
    "D3_mbb_tau_r", "D3_mbb_block_length_time",
    "D3_mbb_effective_blocks",
    "D3_time_sum", "D2_time_sum", "pair_count_time",
]
missing = [key for key in required if key not in result_eulerian]
if missing:
    raise RuntimeError(f"Missing Eulerian fields: {missing}")

for key in required:
    print(f"{key:34s} {np.asarray(result_eulerian[key]).shape}")

r_req = np.asarray(result_eulerian["r_requested"], float).ravel()
r_act = np.asarray(result_eulerian["r_actual"], float).ravel()
count = np.asarray(result_eulerian["count_total"], float).ravel()

if len(r_req) != 12:
    raise RuntimeError(f"Expected 12 scales, got {len(r_req)}")
if np.any(~np.isfinite(r_act)):
    raise RuntimeError("Some Eulerian scales have no realizable grid displacement.")
if np.any(count <= 0):
    raise RuntimeError("Some Eulerian scales have zero samples.")

print("\nrequested r / actual grid r / samples")
for a, b, n in zip(r_req, r_act, count.astype(np.int64)):
    print(f"{a:12.6g}  {b:12.6g}  {n:14d}")
print("\nEulerian calculation passed the structural checks.")

print("\nscale / tau / block length / effective blocks")
for rr, tau, length, neff in zip(
    result_eulerian["dist_axis"],
    result_eulerian["D3_mbb_tau_r"],
    result_eulerian["D3_mbb_block_length_time"],
    result_eulerian["D3_mbb_effective_blocks"],
):
    print(f"{rr:10.5g}  {tau:10.5g}  {int(length):8d}  {int(neff):8d}")


r_requested                        (12,)
r_actual                           (12,)
dist_axis                          (12,)
count_total                        (12,)
pair_count                         (12, 16)
D2                                 (12,)
D3                                 (12,)
H3                                 (12,)
third_count                        (12,)
third_positive_count               (12,)
third_negative_count               (12,)
third_relative_standard_error      (12,)
third_cancellation_ratio           (12,)
dl_reservoir                       (12, 100000)
D3_mbb                             (12, 1000)
D3_mbb_mean                        (12,)
D3_mbb_stderr                      (12,)
D3_mbb_ci_low                      (12,)
D3_mbb_ci_high                     (12,)
D3_mbb_tau_r                       (12,)
D3_mbb_block_length_time           (12,)
D3_mbb_effective_blocks            (12,)
D3_time_sum                        (101, 12)
D2_time_sum                        (10

Module file:
/meddy/simingzhang/Analysis/python/python3/hit_eulerian_matching.py

Calculator signature:
(u, v, r_values, dx, dy=None, nblock_y=4, nblock_x=4, periodic=False, directions=None, min_samples_per_block=500, min_valid_blocks=None, time_chunk=50, require_same_block=False, show_progress=True, progress_name='Eulerian homogeneity')

Runner source contains MBB:
False

Current configuration:
snapshot_dt 0.1
do_moving_block_bootstrap True
mbb_num_boot 1000
mbb_confidence 0.95
mbb_block_length_time None
mbb_random_seed 271828
